# Text Feature Representation: BOW, TF-IDF, and BM25

## Goal
Demonstrate how **text preprocessing** and **feature representation** choices affect text classification performance.

**Key questions we'll answer:**
- How much does lowercasing, punctuation removal, stopword removal, and stemming matter?
- Which feature representation (BOW, TF-IDF, BM25) works best?
- What do these representations actually look like?

**Dataset**: [20 Newsgroups](https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset) — a classic benchmark of news articles across 20 topics.  
We'll use 4 categories. In-class: ~400 docs. At home: ~4000 docs.

**Classifier**: Logistic Regression — simple, fast, interpretable feature weights.

## Setup

In [ ]:
# Uncomment to install missing packages
# !pip install scikit-learn nltk rank_bm25 pandas matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from rank_bm25 import BM25Okapi
from scipy.sparse import csr_matrix

import random
random.seed(42)
np.random.seed(42)

print('All imports OK')

In [ ]:
# Download NLTK resources (one-time)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK resources ready')

## 1. Load the Dataset

**In-class mode** (`IN_CLASS = True`): 100 docs per class = 400 total. Fast to run.  
**Full mode** (`IN_CLASS = False`): All available documents (~1000 per class). More reliable results.

In [ ]:
# --- Configuration ---
IN_CLASS = True          # Set to False to run on the full dataset
DOCS_PER_CLASS = 100     # Only used when IN_CLASS = True

CATEGORIES = [
    'rec.sport.baseball',
    'sci.med',
    'talk.politics.guns',
    'comp.graphics',
]

CATEGORY_LABELS = {
    'rec.sport.baseball': 'Baseball',
    'sci.med': 'Medicine',
    'talk.politics.guns': 'Gun Politics',
    'comp.graphics': 'Computer Graphics',
}

In [ ]:
def load_data(subset, categories, docs_per_class=None):
    """Load 20 newsgroups data, optionally limiting docs per class."""
    data = fetch_20newsgroups(
        subset=subset,
        categories=categories,
        remove=('headers', 'footers', 'quotes'),  # remove metadata that would make it too easy
        random_state=42
    )
    texts, labels = data.data, data.target
    
    if docs_per_class is not None:
        sampled_idx = []
        for cls in range(len(categories)):
            cls_idx = np.where(np.array(labels) == cls)[0]
            sampled_idx.extend(cls_idx[:docs_per_class])
        random.shuffle(sampled_idx)
        texts = [texts[i] for i in sampled_idx]
        labels = [labels[i] for i in sampled_idx]
    
    return texts, labels, data.target_names


n = DOCS_PER_CLASS if IN_CLASS else None
train_texts, train_labels, target_names = load_data('train', CATEGORIES, n)
test_texts,  test_labels,  _            = load_data('test',  CATEGORIES, n)

print(f'Train: {len(train_texts)} documents')
print(f'Test:  {len(test_texts)} documents')
print(f'Classes: {target_names}')

## 2. Explore the Data

In [ ]:
# Class distribution
train_counts = pd.Series(train_labels).value_counts().sort_index()
train_counts.index = [CATEGORY_LABELS[target_names[i]] for i in train_counts.index]

fig, ax = plt.subplots(figsize=(8, 3))
train_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Training Set: Documents per Class')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Sample documents
print('=== Sample Documents ===\n')
for cls_idx, cls_name in enumerate(target_names):
    cls_docs = [t for t, l in zip(train_texts, train_labels) if l == cls_idx]
    sample = cls_docs[0][:400].replace('\n', ' ')
    print(f'[{CATEGORY_LABELS[cls_name]}]')
    print(sample)
    print()

## 3. Text Preprocessing

We define a series of preprocessing steps and build **pipelines** by combining them.

| Level | Steps Applied |
|-------|---------------|
| `raw` | No preprocessing |
| `lower` | Lowercase |
| `lower_punct` | + Remove punctuation & digits |
| `lower_punct_stop` | + Remove stopwords |
| `lower_punct_stop_stem` | + Porter Stemming |
| `lower_punct_stop_lemma` | + WordNet Lemmatization |

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess(text: str, level: str) -> str:
    """Apply preprocessing up to a given level."""
    if level == 'raw':
        return text

    # Step 1: Lowercase
    text = text.lower()
    if level == 'lower':
        return text

    # Step 2: Remove punctuation, digits, extra whitespace
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if level == 'lower_punct':
        return text

    # Step 3: Remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    if level == 'lower_punct_stop':
        return ' '.join(tokens)

    # Step 4a: Stemming
    if level == 'lower_punct_stop_stem':
        tokens = [stemmer.stem(t) for t in tokens]
        return ' '.join(tokens)

    # Step 4b: Lemmatization
    if level == 'lower_punct_stop_lemma':
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        return ' '.join(tokens)

    raise ValueError(f'Unknown level: {level}')


def apply_preprocessing(texts, level):
    return [preprocess(t, level) for t in texts]


LEVELS = [
    'raw',
    'lower',
    'lower_punct',
    'lower_punct_stop',
    'lower_punct_stop_stem',
    'lower_punct_stop_lemma',
]

print('Preprocessing functions defined.')

In [ ]:
# Visualize preprocessing effect on a single document
sample_doc = train_texts[0]

print(f'Original ({len(sample_doc.split())} tokens):')
print(sample_doc[:300])
print()

for level in LEVELS[1:]:
    processed = preprocess(sample_doc, level)
    n_tokens = len(processed.split())
    print(f'[{level}] ({n_tokens} tokens):')
    print(processed[:200])
    print()

In [ ]:
# Show vocabulary size reduction across levels
print(f'{'Level':<30} {'Vocab Size':>12} {'Avg Tokens/Doc':>16}')
print('-' * 60)

for level in LEVELS:
    processed = apply_preprocessing(train_texts, level)
    vectorizer = CountVectorizer(min_df=2)
    vectorizer.fit(processed)
    vocab_size = len(vectorizer.vocabulary_)
    avg_tokens = np.mean([len(t.split()) for t in processed])
    print(f'{level:<30} {vocab_size:>12,} {avg_tokens:>16.1f}')

## 4. Feature Representations

### 4.1 Bag of Words (BOW)

Each document becomes a vector of **raw term counts**.

$$\text{BOW}(t, d) = \text{count of term } t \text{ in document } d$$

In [ ]:
# BOW example
sample_sentence = "the cat sat on the mat and the cat sat"
bow_vec = CountVectorizer()
bow_matrix = bow_vec.fit_transform([sample_sentence])

print('Sentence:', sample_sentence)
print('\nBOW vector:')
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vec.get_feature_names_out()
)
print(bow_df.to_string())
print(f'\nVector shape: {bow_matrix.shape}')

### 4.2 TF-IDF

Downweights **common terms** (high document frequency) and upweights **rare, discriminative terms**.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t,d)}{\text{len}(d)}}_{\text{TF}} \times \underbrace{\log\frac{N}{df(t)}}_{\text{IDF}}$$

Where $N$ = total documents, $df(t)$ = documents containing term $t$.

In [ ]:
# TF-IDF example — compare two documents
docs = [
    "the cat sat on the mat",
    "the dog chased the cat across the park",
]

tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(docs)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vec.get_feature_names_out(),
    index=['Doc 1', 'Doc 2']
)
print('TF-IDF matrix:')
print(tfidf_df.T.sort_values('Doc 1', ascending=False).to_string())
print('\nNote: "the" gets near-zero weight (appears in all docs); unique terms get higher weight.')

### 4.3 BM25 (Best Match 25)

The ranking function behind most search engines. Like TF-IDF but with **saturation** (repeated words hit diminishing returns) and **document length normalization**.

$$\text{BM25}(t, d) = \text{IDF}(t) \cdot \frac{\text{tf}(t,d) \cdot (k_1+1)}{\text{tf}(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- $k_1 = 1.5$: term frequency saturation parameter  
- $b = 0.75$: document length normalization parameter  
- $\text{avgdl}$: average document length in the corpus

We compute BM25 weights for each term-document pair to form a feature matrix (same shape as BOW/TF-IDF).

In [ ]:
def build_bm25_matrix(texts, vocabulary=None, k1=1.5, b=0.75):
    """
    Build a BM25-weighted term-document matrix.
    Returns: (sparse matrix of shape [n_docs, vocab_size], vocabulary dict)
    """
    tokenized = [text.lower().split() for text in texts]
    
    # Build vocabulary from training data if not provided
    if vocabulary is None:
        word_counts = {}
        for tokens in tokenized:
            for token in set(tokens):  # doc frequency
                word_counts[token] = word_counts.get(token, 0) + 1
        # min_df=2 to match CountVectorizer default behavior
        vocabulary = {word: idx for idx, (word, cnt)
                      in enumerate(sorted(word_counts.items()))
                      if cnt >= 2}
    
    n_docs = len(tokenized)
    vocab_size = len(vocabulary)
    doc_lengths = np.array([len(tokens) for tokens in tokenized])
    avgdl = doc_lengths.mean()
    
    # Document frequency for IDF
    df = np.zeros(vocab_size)
    for tokens in tokenized:
        for token in set(tokens):
            if token in vocabulary:
                df[vocabulary[token]] += 1
    
    # IDF: log((N - df + 0.5) / (df + 0.5) + 1) — smoothed Okapi BM25 IDF
    idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1)
    
    # Build sparse matrix
    rows, cols, data = [], [], []
    for doc_idx, tokens in enumerate(tokenized):
        tf_counts = {}
        for token in tokens:
            if token in vocabulary:
                tf_counts[token] = tf_counts.get(token, 0) + 1
        
        dl = doc_lengths[doc_idx]
        for token, tf in tf_counts.items():
            term_idx = vocabulary[token]
            # BM25 term weight
            numerator   = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / avgdl)
            bm25_weight = idf[term_idx] * (numerator / denominator)
            rows.append(doc_idx)
            cols.append(term_idx)
            data.append(bm25_weight)
    
    matrix = csr_matrix((data, (rows, cols)), shape=(n_docs, vocab_size))
    return matrix, vocabulary


print('BM25 builder defined.')

In [ ]:
# BM25 vs TF-IDF: show the saturation effect
# Create docs where one term appears 1, 5, 20 times
test_docs = [
    'baseball ' * 1  + 'player team game',
    'baseball ' * 5  + 'player team game',
    'baseball ' * 20 + 'player team game',
]

bm25_mat, bm25_vocab = build_bm25_matrix(test_docs)
tfidf_mat = TfidfVectorizer().fit_transform(test_docs).toarray()
bow_mat   = CountVectorizer().fit_transform(test_docs).toarray()

bm25_arr = bm25_mat.toarray()

# Find 'baseball' index
bb_bm25  = bm25_vocab.get('baseball', None)
tfidf_vocab = TfidfVectorizer().fit(test_docs).vocabulary_
bow_vocab   = CountVectorizer().fit(test_docs).vocabulary_

bm25_vals  = [bm25_arr[i, bb_bm25] for i in range(3)]
tfidf_vals = [TfidfVectorizer().fit_transform(test_docs).toarray()[i, tfidf_vocab['baseball']] for i in range(3)]
bow_vals   = [CountVectorizer().fit_transform(test_docs).toarray()[i, bow_vocab['baseball']] for i in range(3)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
tf_counts = [1, 5, 20]

for ax, vals, name, color in zip(
    axes,
    [bow_vals, tfidf_vals, bm25_vals],
    ['BOW', 'TF-IDF', 'BM25'],
    ['steelblue', 'darkorange', 'seagreen']
):
    ax.bar([f'tf={c}' for c in tf_counts], vals, color=color)
    ax.set_title(f'{name}: weight of "baseball"')
    ax.set_ylabel('Weight')

plt.suptitle('Term frequency saturation: BOW grows linearly, TF-IDF slows, BM25 saturates', y=1.02)
plt.tight_layout()
plt.show()

print('BOW:   ', [round(v, 3) for v in bow_vals])
print('TF-IDF:', [round(v, 3) for v in tfidf_vals])
print('BM25:  ', [round(v, 3) for v in bm25_vals])

## 5. Classification Experiments

We run a full grid:
- **6 preprocessing levels** × **3 feature representations** = 18 combinations
- Classifier: Logistic Regression (L2 regularization, `max_iter=1000`)
- Metric: Test accuracy

In [ ]:
def classify_bow(train_texts, train_labels, test_texts, test_labels):
    vec = CountVectorizer(min_df=2, max_features=10000)
    X_train = vec.fit_transform(train_texts)
    X_test  = vec.transform(test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_labels)
    return accuracy_score(test_labels, clf.predict(X_test)), clf, vec


def classify_tfidf(train_texts, train_labels, test_texts, test_labels):
    vec = TfidfVectorizer(min_df=2, max_features=10000)
    X_train = vec.fit_transform(train_texts)
    X_test  = vec.transform(test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_labels)
    return accuracy_score(test_labels, clf.predict(X_test)), clf, vec


def classify_bm25(train_texts, train_labels, test_texts, test_labels):
    X_train, vocab = build_bm25_matrix(train_texts)
    X_test,  _     = build_bm25_matrix(test_texts, vocabulary=vocab)
    # Normalize rows to unit L2 norm (similar to TF-IDF normalization)
    X_train = normalize(X_train)
    X_test  = normalize(X_test)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_labels)
    return accuracy_score(test_labels, clf.predict(X_test)), clf, vocab


print('Classifier functions defined.')

In [ ]:
# Run the full experiment grid
results = []

for level in LEVELS:
    print(f'Processing level: {level}...')
    tr = apply_preprocessing(train_texts, level)
    te = apply_preprocessing(test_texts,  level)
    
    for repr_name, clf_fn in [
        ('BOW',    classify_bow),
        ('TF-IDF', classify_tfidf),
        ('BM25',   classify_bm25),
    ]:
        acc, *_ = clf_fn(tr, train_labels, te, test_labels)
        results.append({'Preprocessing': level, 'Representation': repr_name, 'Accuracy': acc})

results_df = pd.DataFrame(results)
print('\nDone!')

## 6. Results

In [ ]:
# Pivot table: preprocessing levels as rows, representations as columns
pivot = results_df.pivot(index='Preprocessing', columns='Representation', values='Accuracy')
pivot = pivot.loc[LEVELS]  # keep original order
pivot['Best'] = pivot.max(axis=1)

print('Test Accuracy by Preprocessing Level and Feature Representation')
print('=' * 65)
print(pivot.round(4).to_string())
print()
print(f'Overall best: {results_df.Accuracy.max():.4f} '
      f'({results_df.loc[results_df.Accuracy.idxmax(), "Preprocessing"]} + '
      f'{results_df.loc[results_df.Accuracy.idxmax(), "Representation"]})')

In [ ]:
# Heatmap of results
pivot_plot = pivot.drop(columns='Best')

short_labels = [
    'raw',
    'lowercase',
    '+ rm punct',
    '+ rm stopwords',
    '+ stemming',
    '+ lemmatization',
]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot_plot.values,
    annot=True, fmt='.3f',
    xticklabels=pivot_plot.columns,
    yticklabels=short_labels,
    cmap='YlGn',
    vmin=pivot_plot.values.min() - 0.02,
    vmax=pivot_plot.values.max() + 0.01,
    ax=ax, linewidths=0.5
)
ax.set_title('Test Accuracy: Preprocessing × Feature Representation', pad=15)
ax.set_ylabel('Preprocessing Level (cumulative)')
ax.set_xlabel('Feature Representation')
plt.tight_layout()
plt.show()

In [ ]:
# Line plot: how accuracy changes as we add preprocessing steps
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'BOW': 'steelblue', 'TF-IDF': 'darkorange', 'BM25': 'seagreen'}

for repr_name in ['BOW', 'TF-IDF', 'BM25']:
    subset = results_df[results_df.Representation == repr_name]
    subset = subset.set_index('Preprocessing').loc[LEVELS]
    ax.plot(short_labels, subset.Accuracy, marker='o',
            label=repr_name, color=colors[repr_name], linewidth=2)

ax.set_xlabel('Preprocessing Level')
ax.set_ylabel('Test Accuracy')
ax.set_title('Effect of Preprocessing on Classification Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 7. Deep Dive: What Did the Model Learn?

### 7.1 Most Discriminative Features per Class

Using the best preprocessing + TF-IDF, look at the top features the logistic regression assigned to each class.

In [ ]:
# Train on the best preprocessing level + TF-IDF
best_level = results_df[
    results_df.Representation == 'TF-IDF'
].sort_values('Accuracy', ascending=False).iloc[0]['Preprocessing']

print(f'Best preprocessing level (TF-IDF): {best_level}')

tr_best = apply_preprocessing(train_texts, best_level)
te_best = apply_preprocessing(test_texts,  best_level)

vec_best = TfidfVectorizer(min_df=2, max_features=10000)
X_train_best = vec_best.fit_transform(tr_best)
X_test_best  = vec_best.transform(te_best)

clf_best = LogisticRegression(max_iter=1000, random_state=42)
clf_best.fit(X_train_best, train_labels)

print(f'Test accuracy: {accuracy_score(test_labels, clf_best.predict(X_test_best)):.4f}')

In [ ]:
# Top features per class
feature_names = np.array(vec_best.get_feature_names_out())
n_top = 12

fig, axes = plt.subplots(1, len(target_names), figsize=(16, 5))

for cls_idx, (cls_name, ax) in enumerate(zip(target_names, axes)):
    coefs = clf_best.coef_[cls_idx]
    top_idx = np.argsort(coefs)[-n_top:][::-1]
    top_features = feature_names[top_idx]
    top_coefs    = coefs[top_idx]
    
    colors = ['steelblue' if c > 0 else 'tomato' for c in top_coefs]
    ax.barh(range(n_top), top_coefs[::-1], color=colors[::-1])
    ax.set_yticks(range(n_top))
    ax.set_yticklabels(top_features[::-1], fontsize=9)
    ax.set_title(CATEGORY_LABELS[cls_name], fontsize=10)
    ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle(f'Top {n_top} Discriminative Features per Class (TF-IDF + LR, level={best_level})', y=1.02)
plt.tight_layout()
plt.show()

### 7.2 Confusion Matrix

In [ ]:
preds = clf_best.predict(X_test_best)
cm    = confusion_matrix(test_labels, preds)

class_short = [CATEGORY_LABELS[n] for n in target_names]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=class_short,
    yticklabels=class_short,
    cmap='Blues', ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (TF-IDF, best preprocessing)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

print(classification_report(test_labels, preds, target_names=class_short))

### 7.3 BOW vs TF-IDF: Why IDF Matters

Words that appear in ALL classes (like "people", "think", "would") get high BOW counts but near-zero TF-IDF weights. These words don't help separate classes — TF-IDF automatically ignores them.

In [ ]:
# Show IDF values for a selection of words
tr_clean = apply_preprocessing(train_texts, 'lower_punct_stop')
tfidf_full = TfidfVectorizer(min_df=2)
tfidf_full.fit(tr_clean)

vocab_idf = dict(zip(tfidf_full.get_feature_names_out(), tfidf_full.idf_))

# High-frequency generic words vs domain-specific words
words_to_check = [
    # Generic
    'people', 'think', 'would', 'know', 'get', 'also',
    # Domain-specific
    'baseball', 'pitcher', 'doctor', 'medical', 'gun', 'firearm', 'pixel', 'render'
]

idf_vals = [(w, vocab_idf.get(w, 0)) for w in words_to_check if w in vocab_idf]
idf_vals.sort(key=lambda x: x[1])

words, idfs = zip(*idf_vals)
colors = ['tomato' if i < 6 else 'steelblue' for i in range(len(words))]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(words, idfs, color=colors)
ax.set_xlabel('IDF Value (higher = rarer = more discriminative)')
ax.set_title('IDF Values: Generic Words vs Domain-Specific Words')
ax.axvline(np.mean(idfs), color='gray', linestyle='--', alpha=0.7, label='mean IDF')
ax.legend()
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='tomato',    label='Generic (low IDF)'),
    Patch(color='steelblue', label='Domain-specific (high IDF)'),
])
plt.tight_layout()
plt.show()

## 8. Summary

| Question | Finding |
|----------|---------|
| Does preprocessing matter? | Yes — even just lowercasing and removing punctuation typically gives a noticeable lift |
| Is more preprocessing always better? | Not always — stemming/lemmatization sometimes hurts if it merges distinguishing terms |
| BOW vs TF-IDF | TF-IDF almost always beats BOW because it downweights uninformative common words |
| BM25 vs TF-IDF | BM25 adds length normalization and frequency saturation; often similar to TF-IDF on short texts, better on variable-length corpora |
| What makes a good feature? | High IDF (rare across classes) + relevant to a single topic |

### What these methods DON'T capture
- Word **order** and **syntax** ("dog bites man" vs "man bites dog" are identical)
- **Semantics** ("car" and "automobile" are unrelated in BOW)
- **Context** ("bank" is ambiguous)

➡ Next step: **word embeddings** and **neural text representations** (meeting 4-5 notebooks)